# Carga de datos

In [1]:
library(grid)
library(dplyr)
library(gridExtra)
library(visualizeR)
library(downscaleR)
library(transformeR)
library(RColorBrewer)
library(latticeExtra)
library(easyVerification)

color = colorRampPalette(rev(brewer.pal(n = 9, "RdYlBu")))


Attaching package: ‘dplyr’


The following objects are masked from ‘package:stats’:

    filter, lag


The following objects are masked from ‘package:base’:

    intersect, setdiff, setequal, union



Attaching package: ‘gridExtra’


The following object is masked from ‘package:dplyr’:

    combine


Loading required package: transformeR




    _______   ____  ___________________  __  ________ 
   / ___/ /  / /  |/  / __  /_  __/ __/ / / / / __  / 
  / /  / /  / / /|_/ / /_/ / / / / __/ / /_/ / /_/_/  
 / /__/ /__/ / /  / / __  / / / / /__ /___  / / \ \ 
 \___/____/_/_/  /_/_/ /_/ /_/  \___/    /_/\/   \_\ 
 
      github.com/SantanderMetGroup/climate4R



transformeR version 2.2.2 (2023-10-26) is loaded


Get the latest stable version (2.2.5) using <devtools::install_github('SantanderMetGroup/transformeR')>

Please see 'citation("transformeR")' to cite this package.

visualizeR version 1.6.4 (2023-10-26) is loaded

Please see 'citation("visualizeR")' to cite this package.

downscaleR version 3.3.4 (2023-06-22) is loaded

Please use 'citation("downscaleR")' to cite this package.

Loading required package: lattice

Loading required package: SpecsVerification


Attaching package: ‘easyVerification’


The following object is masked from ‘package:SpecsVerification’:

    EnsCorr




El primer paso es preparar los datos de nuestro predictando, la temperatura media (tas) de ERA5-Land a 0.1º, y los datos de nuestros predictores, la temperatura media (tas) y la presión en superficie (sp) de ERA5 a 0.25º, pero habiendo escalado los datos a la resolución de nuestro modelo del ECMWF, en este caso 1º.

In [2]:
# Predictando (Y) - ERA5-Land (Alta Resolución 0.1°)
y_obs = readRDS('../../data/downscaling/tas_cgdds_ERA5-Land.rds')
yT_obs = subsetGrid(y_obs, years = 1981:2016)  # training

# Predictores (X) - ERA5 (Resolución Original 0.25º - Interpolada a Resolución SEAS5 1º)
x_sp = readRDS('../../data/downscaling/sp_ERA5.rds')
xT_sp = subsetGrid(x_sp, years = 1981:2016)  # training

x_tas = readRDS('../../data/downscaling/tas_ERA5.rds')
xT_tas = subsetGrid(x_tas, years = 1981:2016)  # training

In [3]:
# Cargo la tas de SEAS5 a 1º
x_tas_seas5 = readRDS('../../data/downscaling/tas_cgdds_seas5_downscaling.rds')

# Renombro variables para que coincidan con ERA5
attr(x_tas_seas5$Variable, "varName") = "t2m"
x_tas_seas5$Variable$varName = "t2m"

# Subset temporal
xT_tas_seas5 = subsetGrid(x_tas_seas5, years = 1981:2016)  # training

# Cargo la sp de SEAS5 a 1º
x_sp_seas5 = readRDS('../../data/downscaling/psl_cgdds_seas5_downscaling.rds')

# Renombro variables para que coincidan con ERA5
attr(x_sp_seas5$Variable, "varName") = "sp"
x_sp_seas5$Variable$varName = "sp"

# Subset temporal
xT_sp_seas5 = subsetGrid(x_sp_seas5, years = 1981:2016)  # training

# Climatología de la temperatura de ERA5-Land

In [4]:
# Valor medio
mean_ref = spatialPlot(climatology(yT_obs),
                       backdrop.theme = "countries",
                       main = "Mean (train)",
                       col.regions = color) %>% suppressMessages %>% suppressWarnings

# Percentil 05
p5_fun = function(x, ...) quantile(x, probs = 0.05, na.rm = TRUE)
p5 = climatology(yT_obs, clim.fun = list(FUN = p5_fun, na.rm = TRUE)) %>% suppressMessages %>% suppressWarnings

p5_ref = spatialPlot(climatology(p5),
                     backdrop.theme = "countries",
                     main = "P05 (train)",
                     col.regions = color) %>% suppressMessages %>% suppressWarnings

# Percentil 95
p95_fun = function(x, ...) quantile(x, probs = 0.95, na.rm = TRUE)
p95 = climatology(yT_obs, clim.fun = list(FUN = p95_fun, na.rm = TRUE)) %>% suppressMessages %>% suppressWarnings

p95_ref = spatialPlot(climatology(p95),
                      backdrop.theme = "countries",
                      main = "P95 (train)",
                      col.regions = color) %>% suppressMessages %>% suppressWarnings

In [5]:
png("metricas_ERA5-Land.png", width = 2000, height = 1000, res = 150)

titulo_fila1 = textGrob("tas (ºC) ERA5-Land (0.1º)",
                        gp = gpar(fontsize = 18, fontface = "bold"))

grid.arrange(titulo_fila1,
             arrangeGrob(mean_ref, p5_ref, p95_ref, ncol = 3),
             ncol = 1,
             heights = c(0.1, 1))

dev.off()

pdf 
  2

In [6]:
# Media del grid para p5
mean_p5 = mean(apply(p5$Data, c(2,3), function(x) mean(x, na.rm = TRUE)), na.rm = TRUE)

# Media del grid para p95
mean_p95 = mean(apply(p95$Data, c(2,3), function(x) mean(x, na.rm = TRUE)), na.rm = TRUE)

# Media del grid para yT_obs
mean_yT = mean(apply(yT_obs$Data, c(2,3), function(x) mean(x, na.rm = TRUE)), na.rm = TRUE)

# Mostrar resultados
cat("Media del grid (ºC):\n",
    "p5: ", mean_p5, "\n",
    "p95: ", mean_p95, "\n",
    "yT_obs: ", mean_yT, "\n")

Media del grid (ºC):
 p5:  9.861209 
 p95:  26.62323 
 yT_obs:  19.19132 


# Entrenamiento del modelo

In [7]:
# Crear directorio temporal para guardar los años sueltos
dir.create("predicciones_tmp", showWarnings = FALSE)

# Identificar los años sobre los que vamos a iterar
years_to_predict = unique(getYearsAsINDEX(xT_tas_seas5))

# Crear una lista vacía para guardar las predicciones de cada paso
pred_list = list()

for (yr in years_to_predict) {

    # Archivo de salida para este año
    f_out = paste0("predicciones_tmp/pred_year_", yr, ".rds")
  
    # Si ya existe, saltar
    if (file.exists(f_out)) {
        message(paste("Año", yr, "ya procesado. Saltando..."))
        next
    }
  
    message(paste("Procesando año:", yr))
  
    # --- DEFINIR TRAINING ---
    # Excluimos el año actual 'yr' de los predictores (ERA5) y del predictando (ERA5-Land)
    xT_tas_train = subsetGrid(xT_tas, years = setdiff(unique(getYearsAsINDEX(xT_tas)), as.integer(yr)))
    xT_sp_train = subsetGrid(xT_sp, years = setdiff(unique(getYearsAsINDEX(xT_sp)), as.integer(yr)))
    
    yT_train = subsetGrid(yT_obs, years = setdiff(unique(getYearsAsINDEX(yT_obs)), as.integer(yr)))
  
    # --- DEFINIR TEST ---
    # Seleccionamos solo el año actual de los datos del modelo estacional
    xT_tas_seas5_target = subsetGrid(xT_tas_seas5, years = as.integer(yr))
    xT_sp_seas5_target = subsetGrid(xT_sp_seas5, years = as.integer(yr))

    # --- ESTANDARIZAR ---
    # Estandarizar SEAS5
    xT_tas_seas5_scaled = scaleGrid(grid = xT_tas_seas5_target,
                                    base = xT_tas_seas5,
                                    ref = xT_tas_train,
                                    by.member = FALSE,
                                    type = "center",
                                    spatial.frame = "gridbox",
                                    time.frame = "monthly") %>% suppressMessages() %>% suppressWarnings()
    
    xT_sp_seas5_scaled = scaleGrid(grid = xT_sp_seas5_target,
                                   base = xT_sp_seas5,
                                   ref = xT_sp_train,
                                   by.member = FALSE,
                                   type = "center",
                                   spatial.frame = "gridbox",
                                   time.frame = "monthly") %>% suppressMessages() %>% suppressWarnings()

    # Unimos los grid con makeMultiGri
    xT_train = makeMultiGrid(xT_tas_train, xT_sp_train) %>% redim(drop = TRUE)
    xT_target = makeMultiGrid(xT_tas_seas5_scaled, xT_sp_seas5_scaled) %>% redim(drop = TRUE)

    # --- PREPARAR DATOS (PCA) ---
    # Calculamos la PCA solo con los datos de entrenamiento para evitar fugas de información
    data_train = prepareData(x = xT_train, 
                             y = yT_train,
                             spatial.predictors = list(
                                 v.exp = 0.95,
                                 which.combine = getVarNames(xT_train))) %>% suppressMessages() %>% suppressWarnings()
  
    # --- ENTRENAR EL MODELO (ANÁLOGOS) ---
    model = downscaleTrain(obj = data_train,
                           method = "analogs", 
                           n.analogs = 1)
  
    # --- PREDICCIÓN (DOWNSCALING) ---
    # Aplicamos el modelo entrenado sobre los datos de SEAS5 (xT_target).
    # climate4r proyectará automáticamente xT_target en las PCs calculadas en data_train.
    
    newdata = prepareNewData(xT_target, data_train)
    pred = downscalePredict(newdata = newdata, model = model)
    pred_final = aggregateGrid(pred, aggr.y = list(FUN = "mean", na.rm = TRUE)) %>% suppressMessages %>% suppressWarnings
    
    # --- GUARDAR Y LIMPIAR ---
  
    # Guardamos el resultado de este año en disco
    saveRDS(pred_final, file = f_out)
  
    # Borramos TODO lo que no sea necesario para la siguiente vuelta
    rm(pred_final, pred, newdata, model, data_train, xT_train, xT_target, yT_train)
  
    # Forzamos al Garbage Collector a liberar la RAM ahora mismo
    gc() 
}

Procesando año: 1981



[2026-01-29 13:20:41.475507] Performing annual aggregation...



[2026-01-29 13:21:04.398086] Done.



Procesando año: 1982



[2026-01-29 13:23:02.263759] Performing annual aggregation...



...

[2026-01-29 14:41:05.53694] Performing annual aggregation...



[2026-01-29 14:41:27.952426] Done.



Procesando año: 2016



[2026-01-29 14:43:31.493496] Performing annual aggregation...



[2026-01-29 14:43:54.213976] Done.



In [8]:
# --- UNIR AL FINAL ---

message("Bucle terminado. Uniendo archivos anuales...")

files = list.files("predicciones_tmp", pattern = "\\.rds$", full.names = TRUE)
years_in_files = as.numeric(gsub("\\D", "", basename(files)))
files = files[order(years_in_files)]

message("Cargando todos los archivos anuales...")
list_of_grids = lapply(files, readRDS)

message("Uniendo grid final...")
prediction_final = bindGrid(list_of_grids, dimension = "time")

# Guardar resultado final
saveRDS(prediction_final, 'pred_annual_knn1.rds')

message("¡Proceso terminado con éxito!")

# Limpieza
unlink("predicciones_tmp", recursive = TRUE)

Bucle terminado. Uniendo archivos anuales...



Cargando todos los archivos anuales...



Uniendo grid final...



¡Proceso terminado con éxito!



# Métricas predictando

In [3]:
yT_obs = aggregateGrid(yT_obs, aggr.y = list(FUN = "mean", na.rm = TRUE)) %>% suppressMessages %>% suppressWarnings

In [6]:
prediction_final = readRDS('pred_annual_knn1.rds')

In [7]:
# Calculo del RMSE
bias = veriApply(verifun = "EnsMe", 
                 fcst = prediction_final$Data, 
                 obs = yT_obs$Data, 
                 ensdim = 1, tdim = 2) %>% suppressMessages %>% suppressWarnings

# Reconstrucción del grid
bias_grid = easyVeri2grid(easyVeri.mat = bias, obs.grid = yT_obs, verifun = "EnsMe")

bias_pred = spatialPlot(climatology(bias_grid),
                        backdrop.theme = "countries",
                        col.regions = color,,
                        main = "Bias (prediction) | Mean = 0.1",
                        at = seq(-0.21, 0.21, 0.021)) %>% suppressMessages %>% suppressWarnings

In [8]:
# Función para calcular correlación de Pearson y valores p entre datos de modelo y observaciones en una grilla espacial
# Además, identifica y marca los puntos con correlación estadísticamente significativa según un umbral de p-valor
#
# Args:
#   model_data: objeto con datos del modelo, estructura esperada con dimensión [miembros, tiempo, latitud, longitud]
#   obs_data: objeto con datos observacionales, estructura con dimensión [tiempo, latitud, longitud]
#   ref_grid: objeto referencia con metadatos espaciales y temporales para construir grillas (xyCoords, Variable, Dates)
#   threshold: umbral para marcar significancia estadística (p-valor), default 0.05
#
# Returns:
#   Lista con:
#     - cor: matriz de correlaciones [lat x lon]
#     - pval: matriz de valores p [lat x lon]
#     - pval_grid: objeto tipo "grid" con valores p y metadatos
#     - pts: lista de objetos para graficar puntos de significancia (stippling)

calc_cor_pval_grid = function(model_data, obs_data, ref_grid, threshold = 0.05) {
    
    # Calcular la media del ensamble para cada punto [tiempo, lat, lon]
    ens_mean = apply(model_data$Data, c(2, 3, 4), mean, na.rm = TRUE)
    
    # Dimensiones espaciales (latitud y longitud)
    lat_n = dim(ens_mean)[2]
    lon_n = dim(ens_mean)[3]
  
    # Inicializar matrices vacías para almacenar correlaciones y p-valores
    cor_array = matrix(NA, nrow = lat_n, ncol = lon_n)
    pval_array = matrix(NA, nrow = lat_n, ncol = lon_n)
    
    # Iterar sobre cada punto espacial
    for (i in 1:lat_n) {
        for (j in 1:lon_n) {
            
            # Extraer series temporales de modelo y observaciones para la celda actual
            pred_series = ens_mean[, i, j]
            obs_series = obs_data$Data[, i, j]
      
            # Filtrar índices con datos completos (no NA)
            valid_idx = complete.cases(pred_series, obs_series)
            
            # Solo calcular correlación si hay suficientes datos (mínimo 10)
            if (sum(valid_idx) >= 10) {
                test = cor.test(pred_series[valid_idx], obs_series[valid_idx], method = "pearson")
                cor_array[i, j] = test$estimate  # Coeficiente de correlación
                pval_array[i, j] = test$p.value  # Valor p de la prueba
            }
        }
    }
  
    # Construir un objeto "grid" para los valores p, con metadatos espaciales y temporales
    pval_grid = list()
    pval_grid$Data = pval_array
    attr(pval_grid$Data, "dimensions") = c("lat", "lon")
    pval_grid$xyCoords = ref_grid$xyCoords
    pval_grid$Variable = ref_grid$Variable
    pval_grid$Dates = ref_grid$Dates
    class(pval_grid) = "grid"

    pval_grid$Variable$varName = "p-values"
    attr(pval_grid$Variable, "description") = "Mapa de p-valores"
    attr(pval_grid$Variable, "units") = ""
    attr(pval_grid$Variable, "longname") = "p-values"
    
    # Crear objetos para graficar puntos de significancia estadística (stippling)
    pts = map.stippling(climatology(pval_grid), 
                        threshold = threshold, 
                        condition = "LT", 
                        pch = 19, col = "black", cex = 0.05) %>% suppressMessages() %>% suppressWarnings()
    
    # Devolver lista con resultados y objetos para plot
    return(list(cor = cor_array, pval = pval_array, pval_grid = pval_grid, pts = pts))
}

In [9]:
test_cor = calc_cor_pval_grid(prediction_final, yT_obs, prediction_final)

# Calculo del RMSE
corr = veriApply(verifun = "EnsCorr", 
                 fcst = prediction_final$Data, 
                 obs = yT_obs$Data, 
                 ensdim = 1, tdim = 2) %>% suppressMessages %>% suppressWarnings

# Reconstrucción del grid
corr_grid = easyVeri2grid(easyVeri.mat = corr, obs.grid = yT_obs, verifun = "EnsCorr")

corr_pred = spatialPlot(climatology(corr_grid),
                       backdrop.theme = "countries",
                       sp.layout = list(test_cor$pts),
                       col.regions = color,
                       main = "Corr (prediction) | Mean = 0.23",
                       at = seq(-1, 1, 0.1)) %>% suppressMessages %>% suppressWarnings

In [10]:
mean_pred = spatialPlot(climatology(prediction_final, by.member = FALSE),
                        backdrop.theme = "countries",
                        main = "Mean (prediction)",
                        col.regions = colorRampPalette(rev(brewer.pal(n = 9, "RdYlBu")))) %>% suppressMessages %>% suppressWarnings

In [12]:
png("metricas_prediction_analogs_final.png", width = 2000, height = 1000, res = 150)

titulo_fila1 = textGrob("tas (ºC) SEAS5 downscaled (0.1º) | Analogs (k = 1)",
                        gp = gpar(fontsize = 18, fontface = "bold"))

grid.arrange(titulo_fila1,
             arrangeGrob(mean_pred, bias_pred, corr_pred, ncol = 3),
             ncol = 1,
             heights = c(0.1, 1))

dev.off()

pdf 
  2